# Financial Policy Document Assistant — RAG Exploration

Step-by-step notebook: **Load → Chunk → Embed → Store (Chroma) → Retrieve → Answer**.

Run it from the `exploration/` folder with the project's virtual environment.

## 0. Setup
Install the packages (skip if you already ran `pip install -r requirements.txt`) and load the OpenAI key from `.env`.

In [ ]:
# The app only needs requirements.txt. The notebook's QA chain (section 6) also needs langchain + langchain-classic:
# !pip install langchain langchain-classic langchain-community langchain-text-splitters langchain-openai pymupdf chromadb python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in the project's .env file"

## 1. Document Understanding
Types: PDF, DOCX, TXT, Confluence, web scraping.

PDF loaders: `pypdf`, `pdfplumber`, `pdfminer`, `pymupdf` / `pymupdf4llm`. We use **PyMuPDF**.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

PDF_PATH = "../data/sample_pdfs/6_financial-policies-and-procedures_2022.pdf"

loader = PyMuPDFLoader(PDF_PATH)
docs = loader.load()
len(docs)

In [ ]:
docs[0]

In [ ]:
print(docs[0].page_content)

In [ ]:
print(docs[1].page_content)

In [ ]:
print(docs[2].page_content)

### Document Object
Each page is a `Document(metadata={...}, page_content="...")`. Metadata includes `source`, `page` (0-based), `total_pages`, `title`, `author`, etc.

## 2. Document Splitting / Chunking
Separators, in order of preference:
- Paragraph — `\n\n`
- New line — `\n`
- Sentence level — `.`
- Word — `" "`

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""],
)

chunks = text_splitter.split_documents(docs)
len(chunks)

In [ ]:
print(chunks[0].page_content)

In [ ]:
print(chunks[1].page_content)

In [ ]:
print(chunks[2].page_content)

## 3. Vector Representation of Text — Embedding Model
1. Open-source embedding — Sentence Transformers `all-MiniLM-L6-v2`
2. OpenAI embedding — ada-002, text-embedding-3-small / large
3. Google embedding — Gemini text embedding

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-ada-002")

## 4. Vector Store — Chroma

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./db",
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

# Alternative: Maximal Marginal Relevance for more diverse chunks
# retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3})

## 5. LLM

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-5.5", temperature=0)

# Equivalent:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-5.5", temperature=0)

## 6. Retrieval QA

In [ ]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_llm(llm, retriever=retriever)

In [ ]:
response = qa_chain.invoke({"query": "What is the main purpose of the Financial Policy and Procedures manual?"})
print(response["result"])

In [ ]:
response = qa_chain.invoke({"query": "What types of bank accounts does Integrity Action maintain?"})
print(response["result"])

In [ ]:
response = qa_chain.invoke({"query": "What are the quotation requirements based on purchase value?"})
print(response["result"])

In [ ]:
response = qa_chain.invoke({"query": "Who are the current authorised bank signatories?"})
print(response["result"])

## Next
The same steps are packaged as modules in `src/` and served by the Streamlit app in `app.py`.